# CIFAR-10 MLP benchmark


### Connexion && Synchronisation Drive - Github

In [ ]:
import os
import json
import urllib.request
from pathlib import Path

BENCHMARK_NAME = 'cifar10'
os.environ['BENCHMARK_NAME'] = BENCHMARK_NAME
SETUP_URL = 'https://raw.githubusercontent.com/Appolinairee/-research-scratch-deep-learning-vision/main/notebooks/colab/setup_colab.ipynb'
SETUP_PATH = Path('/content/setup_colab.ipynb')
urllib.request.urlretrieve(SETUP_URL, SETUP_PATH)
setup_notebook = json.loads(SETUP_PATH.read_text())
for setup_cell in setup_notebook['cells']:
    if setup_cell['cell_type'] == 'code':
        exec(compile(''.join(setup_cell['source']), str(SETUP_PATH), 'exec'), globals())


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from torchvision import datasets

from src.activations import Dropout, ReLU, Softmax
from src.data import augment_images
from src.layers import Dense
from src.losses import MulticlassCrossEntropy
from src.models import Sequential
from src.optim import ReduceLROnPlateau, SGD
from src.regularization import L2Regularization
from src.training.sequential import train_multiclass_classifier

np.set_printoptions(precision=4, suppress=True)

In [ ]:
subprocess.run(
    ['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', 'main'],
    check=True,
)

local_commit = subprocess.check_output(
    ['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'],
    text=True,
).strip()
remote_commit = subprocess.check_output(
    ['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'origin/main'],
    text=True,
).strip()

print('Drive :', local_commit)
print('GitHub:', remote_commit)
print('Aligné :', local_commit == remote_commit)


## CIFAR-10 - Images dataset benchmark

CIFAR-10 et transformer les images en vecteurs.

**Pourquoi** : notre framework actuel sait traiter des couches `Dense`, donc on a besoin d'entrees aplaties.


In [ ]:
train_dataset = datasets.CIFAR10(root=DATA_ROOT, train=True, download=True)
test_dataset = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True)
class_names = train_dataset.classes
class_names

**But** : voir quelques exemples du dataset.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for ax, index in zip(axes.ravel(), range(10)):
    image, label = train_dataset[index]
    ax.imshow(image)
    ax.set_title(class_names[label])
    ax.axis('off')
plt.tight_layout()
plt.show()

But : preparer toutes les images CIFAR-10 en conservant une separation train/validation/test, normaliser et encoder les labels en one-hot.

Pourquoi : utiliser tout le jeu CIFAR-10 sans utiliser le test pour l entrainement ou la selection du modele.


In [ ]:
TRAIN_SAMPLES = 45_000
VALIDATION_SAMPLES = 5_000
TEST_SAMPLES = 10_000
NUM_CLASSES = 10
SEED = 42
AUGMENT_TRAINING = False
AUGMENT_MAX_SHIFT = 4
DROPOUT_RATE = 0.1
PCA_COMPONENTS = 800

rng = np.random.default_rng(SEED)

all_train_indices = rng.permutation(len(train_dataset))
train_indices = all_train_indices[:TRAIN_SAMPLES]
validation_indices = all_train_indices[TRAIN_SAMPLES:]
test_indices = np.arange(len(test_dataset))

def to_numpy_subset(dataset, indices):
    images = []
    labels = []
    for index in indices:
        image, label = dataset[index]
        images.append(np.asarray(image, dtype=np.float32) / 255.0)
        labels.append(label)
    X = np.stack(images)
    y = np.array(labels, dtype=int)
    return X, y

X_train_img, y_train_idx = to_numpy_subset(train_dataset, train_indices)
X_validation_img, y_validation_idx = to_numpy_subset(train_dataset, validation_indices)
X_test_img, y_test_idx = to_numpy_subset(test_dataset, test_indices)
if AUGMENT_TRAINING:
    X_train_img = augment_images(
        X_train_img,
        horizontal_flip=True,
        max_shift=AUGMENT_MAX_SHIFT,
        seed=SEED,
    )

X_train = X_train_img.reshape(len(X_train_img), -1)
X_validation = X_validation_img.reshape(len(X_validation_img), -1)
X_test = X_test_img.reshape(len(X_test_img), -1)

mean = X_train.mean(axis=0, keepdims=True)
std = X_train.std(axis=0, keepdims=True) + 1e-8
X_train = (X_train - mean) / std
X_validation = (X_validation - mean) / std
X_test = (X_test - mean) / std

# PCA et blanchiment ajustes uniquement sur les donnees d entrainement.
pca = PCA(
    n_components=PCA_COMPONENTS,
    whiten=True,
    svd_solver='randomized',
    random_state=SEED,
)
X_train = pca.fit_transform(X_train).astype(np.float32)
X_validation = pca.transform(X_validation).astype(np.float32)
X_test = pca.transform(X_test).astype(np.float32)

y_train = np.eye(NUM_CLASSES)[y_train_idx]
y_validation = np.eye(NUM_CLASSES)[y_validation_idx]
y_test = np.eye(NUM_CLASSES)[y_test_idx]

assert len(train_dataset) == 50_000
assert len(test_dataset) == 10_000
assert len(train_indices) == TRAIN_SAMPLES
assert len(validation_indices) == VALIDATION_SAMPLES
assert len(test_indices) == TEST_SAMPLES
assert set(train_indices).isdisjoint(validation_indices)
assert len(train_indices) + len(validation_indices) == len(train_dataset)
assert X_train.shape == (TRAIN_SAMPLES, PCA_COMPONENTS)
assert X_validation.shape == (VALIDATION_SAMPLES, PCA_COMPONENTS)
assert X_test.shape == (TEST_SAMPLES, PCA_COMPONENTS)
assert y_train.shape == (TRAIN_SAMPLES, NUM_CLASSES)
assert y_validation.shape == (VALIDATION_SAMPLES, NUM_CLASSES)
assert y_test.shape == (TEST_SAMPLES, NUM_CLASSES)
assert np.all((y_train_idx >= 0) & (y_train_idx < NUM_CLASSES))
assert np.all((y_validation_idx >= 0) & (y_validation_idx < NUM_CLASSES))
assert np.all((y_test_idx >= 0) & (y_test_idx < NUM_CLASSES))
assert np.allclose(y_train.sum(axis=1), 1.0)
assert np.allclose(y_validation.sum(axis=1), 1.0)
assert np.allclose(y_test.sum(axis=1), 1.0)

print('Verification checklist:')
print('- CIFAR-10 source sizes: 50,000 train / 10,000 test')
print('- Split: 45,000 train / 5,000 validation / 10,000 test')
print('- All CIFAR-10 training images are used exactly once in train/validation')
print('- Augmentation applied to training images only:', AUGMENT_TRAINING)
print('- PCA whitening components:', PCA_COMPONENTS)
print('- Train and validation indices are disjoint')
print('- Normalization fitted on training data only')
print('- Labels are valid one-hot vectors for 10 classes')

X_train.shape, y_train.shape, X_validation.shape, y_validation.shape, X_test.shape, y_test.shape


**But** : tester une architecture dense inspiree de Z-LIN avec le protocole de baseline.

**Important** : cette cellule active PCA/blanchiment, l'architecture 4000-1000 alternee et le dropout 0,1. Le pre-entrainement ZAE specifique de l'article n'est pas encore implemente.


In [ ]:

INPUT_DIM = PCA_COMPONENTS
NUM_CLASSES = 10
HIDDEN_UNITS = (4000, 1000, 4000, 1000, 4000, 1000, 4000)
LEARNING_RATE = 1e-3
MOMENTUM = 0.9
L2_STRENGTH = 1e-4
EPOCHS = 300
BATCH_SIZE = 100

model_layers = []
previous_dim = INPUT_DIM
for index, hidden_dim in enumerate(HIDDEN_UNITS):
    model_layers.extend([
        Dense(input_dim=previous_dim, output_dim=hidden_dim, seed=index),
        ReLU(),
        Dropout(rate=DROPOUT_RATE, seed=10 + index),
    ])
    previous_dim = hidden_dim
model_layers.extend([
    Dense(input_dim=previous_dim, output_dim=NUM_CLASSES, seed=100),
    Softmax(),
])
model = Sequential(model_layers)

dense_layers = [layer for layer in model.layers if isinstance(layer, Dense)]
actual_architecture = [dense_layers[0].input_dim]
actual_architecture.extend(layer.output_dim for layer in dense_layers)
expected_architecture = [INPUT_DIM, *HIDDEN_UNITS, NUM_CLASSES]
print({
    'expected_architecture': expected_architecture,
    'actual_architecture': actual_architecture,
    'dense_layer_count': len(dense_layers),
})
assert actual_architecture == expected_architecture, (
    f'Architecture mismatch: expected {expected_architecture}, '
    f'got {actual_architecture}. Re-run synchronization and all cells from the beginning.'
)

sample_output = model.forward(X_train[:8])
assert sample_output.shape == (8, NUM_CLASSES)
np.testing.assert_allclose(sample_output.sum(axis=1), 1.0, atol=1e-6)

print({
    'split': (TRAIN_SAMPLES, VALIDATION_SAMPLES, TEST_SAMPLES),
    'architecture': (INPUT_DIM, *HIDDEN_UNITS, NUM_CLASSES),
    'learning_rate': LEARNING_RATE,
    'momentum': MOMENTUM,
    'dropout_rate': DROPOUT_RATE,
    'l2_strength': L2_STRENGTH,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'seed': SEED,
})

loss = MulticlassCrossEntropy()

optimizer = SGD(lr=LEARNING_RATE, momentum=MOMENTUM)
regularizer = L2Regularization(strength=L2_STRENGTH)
model


## Configuration unique du backend

Une seule exécution est lancée : NumPy sur CPU ou CuPy sur GPU selon `BACKEND`.


In [ ]:
import time

from src.core.backend import get_backend

BACKEND = 'gpu'  # Choisir 'cpu' ou 'gpu'.
backend = get_backend(BACKEND)
print({
    'backend': 'CuPy' if backend.is_gpu else 'NumPy',
    'device': 'CUDA' if backend.is_gpu else 'CPU',
    'backend_version': backend.xp.__version__,
})

model_layers = []
previous_dim = INPUT_DIM
for index, hidden_dim in enumerate(HIDDEN_UNITS):
    model_layers.extend([
        Dense(input_dim=previous_dim, output_dim=hidden_dim, seed=index, backend=backend),
        ReLU(backend=backend),
        Dropout(rate=DROPOUT_RATE, seed=10 + index, backend=backend),
    ])
    previous_dim = hidden_dim
model_layers.extend([
    Dense(input_dim=previous_dim, output_dim=NUM_CLASSES, seed=100, backend=backend),
    Softmax(backend=backend),
])
model = Sequential(model_layers)
loss = MulticlassCrossEntropy(backend=backend)
regularizer = L2Regularization(strength=L2_STRENGTH)

sample_output = model.forward(X_train[:8])
assert backend.is_array(sample_output)
assert sample_output.shape == (8, NUM_CLASSES)
probability_sums = backend.sum(sample_output, axis=1)
assert backend.scalar(probability_sums.min()) > 0.999
assert backend.scalar(probability_sums.max()) < 1.001

print({
    'split': (TRAIN_SAMPLES, VALIDATION_SAMPLES, TEST_SAMPLES),
    'architecture': (INPUT_DIM, *HIDDEN_UNITS, NUM_CLASSES),
    'learning_rate': LEARNING_RATE,
    'momentum': MOMENTUM,
    'dropout_rate': DROPOUT_RATE,
    'l2_strength': L2_STRENGTH,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'seed': SEED,
})


**But** : entraîner le modèle une seule fois avec le backend choisi.

**Pourquoi** : comparer uniquement les performances du backend explicitement sélectionné.


In [ ]:
start_time = time.perf_counter()
scheduler = ReduceLROnPlateau(factor=0.5, patience=10, min_lr=1e-6)
history = train_multiclass_classifier(
    model=model,
    X=X_train,
    y=y_train,
    loss=loss,
    optimizer=optimizer,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    validation_data=(X_validation, y_validation),
    regularizer=regularizer,
    scheduler=scheduler,
)
backend.synchronize()
elapsed_seconds = time.perf_counter() - start_time

assert len(history) == EPOCHS
assert all(row.validation_loss is not None for row in history)
assert all(row.validation_accuracy is not None for row in history)
assert all(row.weight_norm is not None for row in history)
assert all(row.gradient_norm is not None for row in history)
assert all(row.relu_zero_fraction is not None for row in history)

best_epoch = history.best_epoch
best_metrics = history[best_epoch]
assert 0 <= best_epoch < EPOCHS

print({
    'backend': 'CuPy' if backend.is_gpu else 'NumPy',
    'device': 'CUDA' if backend.is_gpu else 'CPU',
    'elapsed_seconds': round(elapsed_seconds, 2),
    'best_epoch': best_epoch,
    'best_validation_loss': best_metrics.validation_loss,
    'best_validation_accuracy': best_metrics.validation_accuracy,
})


**But** : visualiser la loss et l'accuracy train.

**Pourquoi** : verifier la convergence du MLP avant l'evaluation finale sur le test set.


In [ ]:
epochs = [row.epoch for row in history]
losses = [row.loss for row in history]
accuracies = [row.accuracy for row in history]
validation_losses = [row.validation_loss for row in history]
validation_accuracies = [row.validation_accuracy for row in history]
weight_norms = [row.weight_norm for row in history]
gradient_norms = [row.gradient_norm for row in history]
relu_zero_fractions = [row.relu_zero_fraction for row in history]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, losses)
axes[0].plot(epochs, validation_losses)
axes[0].legend(['train', 'validation'])
axes[0].set_title('Loss (train / validation)')
axes[0].set_xlabel('epoch')
axes[0].grid(True, alpha=0.25)

axes[1].plot(epochs, accuracies)
axes[1].plot(epochs, validation_accuracies)
axes[1].legend(['train', 'validation'])
axes[1].set_title('Accuracy (train / validation)')
axes[1].set_xlabel('epoch')
axes[1].grid(True, alpha=0.25)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(epochs, weight_norms)
axes[0].set_title('Weight norm')
axes[1].plot(epochs, gradient_norms)
axes[1].set_title('Gradient norm')
axes[2].plot(epochs, relu_zero_fractions)
axes[2].set_title('ReLU zero fraction')
for ax in axes:
    ax.set_xlabel('epoch')
    ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.show()


**But** : evaluer le modele sur le test set.

**Pourquoi** : mesurer la generalisation reelle du MLP from scratch sur CIFAR-10.


In [ ]:
# Evaluation finale sur le test set, sans selectionner de modele avec le test.
from src.metrics import multiclass_accuracy

test_probs = model.predict_proba(X_test)
test_acc = multiclass_accuracy(y_test, test_probs, backend=backend)
test_pred_idx = backend.to_host(backend.argmax(test_probs, axis=1))
test_true_idx = np.argmax(y_test, axis=1)
test_probs = backend.to_host(test_probs)

print('Final held-out test accuracy =', round(test_acc, 4))
print('Test set was not used for epoch or hyperparameter selection.')
print('Reported model: final training epoch; best validation epoch is reported above.')


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

cm = confusion_matrix(test_true_idx, test_pred_idx)
fig, ax = plt.subplots(figsize=(10, 10))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, cmap='Blues', xticks_rotation=45, colorbar=False)
plt.show()

### Comparaison finale CPU / GPU

Le test est évalué après la sélection par validation.


In [ ]:
# Evaluation finale sur le test set, après la sélection par validation.
test_metrics = evaluate_multiclass(model, X_test, y_test, loss)
print({
    'backend': 'CuPy' if backend.is_gpu else 'NumPy',
    'device': 'CUDA' if backend.is_gpu else 'CPU',
    'test_loss': test_metrics.loss,
    'test_accuracy': test_metrics.accuracy,
})
print('Le test n’a pas été utilisé pour sélectionner le modèle.')


**But** : afficher la matrice de confusion.

**Pourquoi** : voir quelles classes le MLP confond le plus.
